#RS Datacatlog Uploader

Data attributes DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537155636

Interaction events DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537352234

In [138]:
import dotenv
from typing import List, Dict
import json
dotenv.load_dotenv()

import os
import requests
import pandas as pd
import re

RS_API_KEY = os.getenv("RUDDERSTACK_API_KEY")
RS_API_URL = "https://api.rudderstack.com"

headers = {
    "Authorization": f"Bearer {RS_API_KEY}",
    "Content-Type": "application/json",
}

data_attributes_filepath = "data/Data attributes.csv" # path to data attributes file exported from confluence
interaction_events_filepath = "data/Interaction events.csv" # path to interaction events file exported from confluence
required_events_filepath = "data/Required_properties.csv" # path to file holding list of required properties for events
tracking_plan_name = "MAC website" # Set this for tracking plan name
category_name = "MAC website"  # Set this for category name put on created events
category_id = "" # Leave blank


In [127]:
# functions
def save_event_id(event_name:str, event_id:str) -> bool:
    """
    Update event in df_event_index with passed event id\n 
    params:\n
      event_name: name of event to update
      param event_id: event_id\n
    returns:\n
      True if event updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if event_name in df_event_index['event'].values:
        df_event_index.loc[df_event_index['event'] == event_name, 'event_id'] = event_id
        rc = True
    
    return rc

def save_property_id(property_name:str, property_type:str, property_id:str) -> bool:
    """
    Update all events in df_event_index that have this property with the passed property id.\n 
    params:\n
      property_name: name of property to update
      property_type: type of property to update
      property_id: id of property\n
    returns:\n
      True if properties updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if not(df_event_index[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type)]).empty:
        df_event_index.loc[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type), 'property_id'] = property_id
        rc = True
    
    return rc

    

In [128]:
# import interaction events file
df_events = pd.read_csv(interaction_events_filepath)
df_events['Trigger'] = df_events['Trigger'].fillna('(blank)') # default empty trigger values
df_events['Trigger'] = df_events['Trigger'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_events['Trigger'] = df_events['Trigger'].str.strip() # remove leading/trailing spaces

# import data attributes file
valid_types = ['object','string','number','integer','array','boolean','null']
df_properties = pd.read_csv(data_attributes_filepath)
df_properties = df_properties[df_properties['Data attribute'].notna()] # remove rows with missing 'Data attribute' values
df_properties['Type'] = df_properties['Type'].str.lower().str.strip() # lowercase all values and remove leading/trailing whitespace 
df_properties['Type'] = df_properties['Type'].replace(['object', 'array', 'list'], 'string') # change 'object', 'array', and 'list'and  types to 'string' 
df_properties['Description'] = df_properties['Description'].fillna('(blank)') # default empty description values
df_properties['Description'] = df_properties['Description'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_properties['Description'] = df_properties['Description'].str.strip() # remove leading/trailing spaces 

# Check properties have valid types, otherwise remove property and log error
invalid_rows = df_properties[~df_properties['Type'].isin(valid_types)]
if len(invalid_rows) > 0:
    print(f"Found {len(invalid_rows)} rows with invalid types:")
    for idx, row in invalid_rows.iterrows():
        print(f"  Row {idx}: '{row['Data attribute']}' has invalid type '{row['Type']}'")
    df_properties = df_properties[df_properties['Type'].isin(valid_types)]


In [ ]:
# Build an indexing list of properties within an event.  Set required to False as default.  

#  Process properties data, capture property type as property name/type is unique.
property_rows = []
for idx, property in df_properties.iterrows():
    event_list = property['Associated with'].split('\n')
    for event in event_list:
        property_rows.append({"event": event, "property": property['Data attribute'], "type": property['Type'], "required":False, "event_id":None, "property_id":None})
    
df_event_index = pd.DataFrame(property_rows)
df_event_index = df_event_index.sort_values(by=['event'])

# Now search event data for any events with no properties against it (e.g. eol_complete) and add them
event_rows = []
for idx, event in df_events.iterrows():
    if pd.isna(event['Parameters']):
        #print(f"{event['Event name']}: {event['Parameters']}")
        event_rows.append({"event": event['Event name'], "property": None,"type":None, "required":None, "event_id":None, "property_id":None})
df_event_extras = pd.DataFrame(event_rows)
#print(df_event_extras)

df_event_index = pd.concat([df_event_index, df_event_extras], ignore_index=True)
df_event_index = df_event_index.sort_values(by=['event']) 

In [139]:
df_required_properties = pd.DataFrame()

# If file exists, update event_index with correct required values (e.g. True = required)
try:
    df_required_properties = pd.read_csv(required_events_filepath )
except FileNotFoundError: 
    print(f"File '{required_events_filepath}' does not exist, skipping updating event_index")
    df_required_properties = pd.DataFrame()

if not df_required_properties.empty:
    # Update event_index with required properties in events 
    for idx, row in df_required_properties.iterrows():
        # use values row['event'] and row['required_property'] to find matching row in df_event_index, e.g.  df_event_index['event'] and df_event_index['property'] columns
        index_event = df_event_index[(df_event_index['event'] == row['event']) & (df_event_index['property'] == row['required_property'])]
        
        #print(index_event)
        if index_event.empty:
            print(f"Event: '{row['event']}', Property: '{row['required_property']}' not found")
        else:
            df_event_index.loc[index_event.index, 'required'] = True
            

Event: 'abandon_tool', Property: 'fake property' not found
Event: 'fake_event', Property: 'tool' not found


In [140]:
# Create category to tag events with
body = {
    "name": "MAC website",
    "description": "Event stream from MAC website"
}

response = requests.post(f"{RS_API_URL}/v2/catalog/categories", headers=headers, json=body )

if (response.status_code == 200):
    print(f"Category created successfully - id: {response.json()['id']}")
    category_id = response.json()['id']
elif (response.status_code == 400):
    print(f"Category already exists - [{response.status_code}] {response.json()['error']}")
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/categories", headers=headers )
    
    for category in response.json()["data"]:
        if category["name"] == body["name"]:
            category_id = category["id"]
            print("Retrieved category ID: " + category_id)
            break   
else:
    print(f"Error creating category - [{response.status_code}] {response.json()['error']}")
    
    

Category already exists - [400] Category with name MAC website already exists
Retrieved category ID: cat_2yyODDUFmTgMo3sOMSf7RSftHpc


In [141]:
# Upload properties to data catalog, if property already exists log it and skip to next record

print("Uploading properties to data catalog...")
count = 0
for idx, row in df_properties.head(10).iterrows():

    body = {
    "name": row['Data attribute'],
    "description": row['Description'],
    "type": row['Type'],
    }

    response = requests.post(f"{RS_API_URL}/v2/catalog/properties", json=body, headers=headers )

    if response.status_code == 200:
        count += 1
        # Save property id to event index
        if not(save_property_id(row['Data attribute'],row['Type'], response.json()['id'])):
            print(f"Error updating event index for property. Name: {row['Data attribute']}, Type: {row['Type']}, ID: {response.json()['id']})")
    else:
        print(f"Error creating property - [{response.status_code}] {response.json()['error']}")

print(f"Created {count}/{len(df_properties)} properties")



Uploading properties to data catalog...
Error creating property - [400] Property with name interaction_type and type string already exists
Error creating property - [400] Property with name parent_title and type string already exists
Error creating property - [400] Property with name detail_title and type string already exists
Error creating property - [400] Property with name component_id and type string already exists
Error creating property - [400] Property with name option_selected and type string already exists
Error creating property - [400] Description must be between 3 and 2000 characters long and start with a letter.
Created 4/86 properties


In [ ]:
# Upload events to data catalog, if event already exists log it and skip to next record
print("Uploading events to data catalog...")

count = 0
body_extras = {}
if category_id != "":
        body_extras[""] = category_id

for idx, row in df_events.head(10).iterrows():

    # create event
    body = {
    "name": row['Event name'],
    "description": row['Trigger'],
    "eventType": "track",
    **body_extras
    }
   

    response = requests.post(f"{RS_API_URL}/v2/catalog/events", json=body, headers=headers )
    
    if response.status_code == 200:
        count += 1
        # save event id to event index
        if not(save_event_id(row['Event name'], response.json()['id'])):
            print(f"Error updating event index with event. Name: {row['Event name']}, ID: {response.json()['id']}")
    else:
        print(f"Error creating event '{row['Event name']}'- [{response.status_code}] {response.json()['error']}")

print(f"Created {count}/{len(df_events)} events")

Uploading events to data catalog...
Error creating event 'component_interaction'- [400] Event with name component_interaction already exists
Error creating event 'wayfinder_start'- [400] Event with name wayfinder_start already exists
Error creating event 'wayfinder_next'- [400] Event with name wayfinder_next already exists
Error creating event 'wayfinder_back'- [400] Event with name wayfinder_back already exists
Error creating event 'wayfinder_complete'- [400] Event with name wayfinder_complete already exists
Error creating event 'outlet_interaction'- [400] Event with name outlet_interaction already exists
Error creating event 'outlet_view'- [400] Event with name outlet_view already exists
Error creating event 'provider_interaction'- [400] Event with name provider_interaction already exists
Created 2/82 events


In [ ]:
# Creating tracking plan

In [ ]:
# Populate tracking plan
